### Update projected starting lineups

In [19]:
from MODELS.scrapStarting import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

Successfully updated /Users/alexgonzalez/Documents/NBA-Prop-Predictor/PRODUCTION/teamInfo.py
Updated 16 teams with confirmed lineups


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from FEATURES.featuresV2 import *
from PRODUCTION.calculateEVS import *
from PRODUCTION.pipeline import *
from PRODUCTION.teamInfo import teamStarPlayer, projectedStartingFive, mainStartingFive

### Load Model

In [2]:
# Load split NGBoost models (mean, variance, calibration factor, and isotonic calibrator)
pts_mean_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_MEAN_MODEL_PRODUCTION.pkl')
pts_var_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_VAR_MODEL_PRODUCTION.pkl')
calibration_factor = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_CALIBRATION_FACTOR_PRODUCTION.pkl')

model = (pts_mean_model, pts_var_model, calibration_factor)  
features = joblib.load('../MODELS/SAVED_MODELS/feature_list.pkl')

print(f"Loaded models with calibration factor: {calibration_factor}")

Loaded models with calibration factor: 4.5


### Load Player Data and Bookmaker Data

In [5]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')

usData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_US_{today}.csv')
dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_{today}.csv')

dfsData.head()

/var/folders/9q/5_554qsx5z70w9d_vkmvjg0h0000gn/T/ipykernel_73473/1321450873.py:5: DtypeWarning: Columns (33) have mixed types. Specify dtype option on import or set low_memory=False.
  s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE
0,Betr DFS,player_points,Miles Bridges,Over,21.5,-137,2025-11-23,2025-11-24T00:32:52Z
1,Betr DFS,player_points,Miles Bridges,Under,21.5,-137,2025-11-23,2025-11-24T00:32:52Z
2,Betr DFS,player_points,Kon Knueppel,Over,25.5,-137,2025-11-23,2025-11-24T00:32:52Z
3,Betr DFS,player_points,Kon Knueppel,Under,25.5,-137,2025-11-23,2025-11-24T00:32:52Z
4,Betr DFS,player_points,Onyeka Okongwu,Over,23.5,-137,2025-11-23,2025-11-24T00:32:52Z


## Top EVs for 2 leg bets

### Underdog picks

In [6]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

underdogPairs = calculate2LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=10)


underdogPairs = underdogPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'PROB 1', 'PROB 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']]
underdogPairs.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogPairs.csv', index=False)
underdogPairs

Pre-computing predictions for 20 players...
Processing 20 players...
Generated 155 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,PREDICTION 1,PREDICTION 2,PROB 1,PROB 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV%,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
42,Rui Hachimura,Bennedict Mathurin,11.5,20.5,15.43,25.31,0.728,0.769,over,over,0,64.75,0.324,High,High
69,Kevin Love,Jalen Duren,4.5,18.5,7.45,22.20,0.726,0.706,over,over,0,50.76,0.254,Low,High
114,Harrison Barnes,Andrew Nembhard,13.5,17.5,16.76,20.73,0.699,0.686,over,over,0,41.08,0.205,High,High
61,Deandre Ayton,Isaiah Jackson,14.5,7.5,15.99,10.10,0.595,0.681,over,over,0,18.98,0.095,High,Med
96,Kyle Filipowski,Duncan Robinson,8.5,10.5,9.62,12.84,0.584,0.649,over,over,0,11.47,0.057,Med,High
9,Luka Dončić,Ausar Thompson,31.5,11.5,32.42,13.77,0.565,0.639,over,over,0,6.11,0.031,Med,High
22,Lauri Markkanen,Pascal Siakam,26.5,23.5,27.39,24.21,0.552,0.542,over,over,0,-12.10,0.000,High,High
110,Svi Mykhailiuk,Jarace Walker,8.5,8.5,9.24,8.18,0.548,0.521,over,under,0,-16.05,0.000,High,Med
83,Brice Sensabaugh,T.J. McConnell,8.5,7.5,8.14,7.21,0.530,0.521,under,under,0,-18.83,0.000,Low,Med
121,Jeremy Sochan,Cade Cunningham,8.5,26.5,8.54,26.63,0.503,0.508,over,over,0,-24.96,0.000,Med,High


### Prizepicks picks

In [7]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

prizepicksPairs = calculate2LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=10)


pairsPrizepicks = prizepicksPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'PROB 1', 'PROB 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
pairsPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksPairs.csv', index=False)
prizepicksPairs

Pre-computing predictions for 40 players...
Error getting prediction for LeBron James: float division by zero
Processing 38 players...
Generated 599 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,ODDS 1,ODDS 2,PREDICTION 1,PREDICTION 2,MODEL SIDE 1,MODEL SIDE 2,PROB 1,PROB 2,PROB BOTH,EDGE 1,EDGE 2,COMBINED EDGE,EV%,KELLY FULL,RECOMMENDATION,SIGMA 1,SIGMA 2,SIGMA FLAG 1,SIGMA FLAG 2,CI 1,CI 2,CORRELATION,SAME_GAME,EXPECTED ROI
426,Dillon Brooks,Bennedict Mathurin,18.5,19.5,-135,-137,23.81,25.31,over,over,0.770,0.813,0.6133,0.193,0.237,0.294,84.00,0.420,1,7.19,6.53,High,High,"(9.7, 37.9)","(12.5, 38.1)",0.05,0,84.0
84,Austin Reaves,Jalen Duren,22.5,18.5,-127,-137,26.89,22.20,over,over,0.730,0.706,0.5050,0.161,0.137,0.192,51.51,0.258,0,7.18,6.82,High,High,"(12.8, 41.0)","(8.8, 35.6)",0.05,0,51.5
188,Rui Hachimura,Harrison Barnes,11.5,13.5,-115,-130,15.43,16.76,over,over,0.728,0.699,0.4990,0.178,0.149,0.207,49.71,0.249,0,6.46,6.25,High,High,"(2.8, 28.1)","(4.5, 29.0)",0.05,0,49.7
330,Jake LaRavia,Tobias Harris,7.5,11.5,-135,-137,11.15,14.81,over,over,0.719,0.689,0.4860,0.143,0.113,0.164,45.79,0.229,0,6.29,6.70,High,High,"(0.0, 23.5)","(1.7, 27.9)",0.05,0,45.8
377,Kevin Love,Andrew Nembhard,5.0,17.5,-137,-137,7.45,20.73,over,over,0.691,0.686,0.4650,0.113,0.108,0.140,39.50,0.197,0,4.91,6.65,Low,High,"(0.0, 17.1)","(7.7, 33.8)",0.05,0,39.5
448,Devin Vassell,Isaiah Jackson,17.5,7.5,-136,-137,14.77,10.10,under,over,0.661,0.681,0.4409,0.084,0.103,0.117,32.27,0.161,0,6.57,5.54,High,Med,"(1.9, 27.7)","(0.0, 21.0)",0.05,0,32.3
116,Keyonte George,Duncan Robinson,20.5,10.5,-130,-137,23.03,12.84,over,over,0.651,0.649,0.4143,0.080,0.078,0.096,24.28,0.121,0,6.52,6.10,High,High,"(10.3, 35.8)","(0.9, 24.8)",0.05,0,24.3
354,Marcus Smart,Ausar Thompson,6.5,11.5,-118,-137,8.76,13.77,over,over,0.647,0.639,0.4055,0.088,0.079,0.101,21.65,0.108,0,5.97,6.37,Med,High,"(0.0, 20.5)","(1.3, 26.3)",0.05,0,21.7
239,Isaiah Collier,Collin Gillespie,8.0,14.5,-137,-125,9.32,12.16,over,under,0.610,0.636,0.3801,0.043,0.069,0.067,14.04,0.070,0,4.74,6.72,Low,High,"(0.0, 18.6)","(0.0, 25.3)",0.05,0,14.0
127,Deandre Ayton,De'Aaron Fox,14.5,23.5,-135,-125,15.99,24.98,over,over,0.595,0.586,0.3412,0.030,0.021,0.029,2.37,0.012,0,6.23,6.85,High,High,"(3.8, 28.2)","(11.6, 38.4)",0.05,0,2.4


## 3 leg parlay

### Underdog picks

In [8]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

underdogTrios = calculate3LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=10)

underdogTrios = underdogTrios[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'PROB 1', 'PROB 2', 'PROB 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
underdogTrios.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogTrios.csv', index=False)
underdogTrios.head()

Pre-computing predictions for 20 players...
Processing 20 players...
Generated 964 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,PROB 1,PROB 2,PROB 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV%,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
354,Rui Hachimura,Harrison Barnes,Bennedict Mathurin,11.5,13.5,20.5,15.43,16.76,25.31,0.728,0.699,0.769,over,over,over,0,111.52,0.223,High,High,High
613,Kevin Love,Andrew Nembhard,Isaiah Jackson,4.5,17.5,7.5,7.45,20.73,10.10,0.726,0.686,0.681,over,over,over,0,83.21,0.166,Low,High,Med
515,Deandre Ayton,Jalen Duren,Duncan Robinson,14.5,18.5,10.5,15.99,22.20,12.84,0.595,0.706,0.649,over,over,over,0,47.18,0.094,High,High,High
202,Lauri Markkanen,Kyle Filipowski,Ausar Thompson,26.5,8.5,11.5,27.39,9.62,13.77,0.552,0.584,0.639,over,over,over,0,11.25,0.022,High,Med,High
121,Luka Dončić,Pascal Siakam,Jarace Walker,31.5,23.5,8.5,32.42,24.21,8.18,0.565,0.542,0.521,over,over,under,0,-13.87,0.000,Med,High,Med


### Prizepicks picks

In [9]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

triosPrizepicks = calculate3LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=15)


triosPrizepicks = triosPrizepicks[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'PROB 1', 'PROB 2', 'PROB 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
triosPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksTrios.csv', index=False)
triosPrizepicks.head()

Pre-computing predictions for 40 players...
Error getting prediction for LeBron James: float division by zero
Processing 38 players...
Generated 7621 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,PROB 1,PROB 2,PROB 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV%,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
1473,Austin Reaves,Dillon Brooks,Bennedict Mathurin,22.5,18.5,19.5,26.89,23.81,25.31,0.730,0.770,0.813,over,over,over,1,146.62,0.293,High,High,High
3280,Rui Hachimura,Jake LaRavia,Jalen Duren,11.5,7.5,18.5,15.43,11.15,22.20,0.728,0.719,0.706,over,over,over,0,99.80,0.200,High,High,High
5907,Kevin Love,Harrison Barnes,Tobias Harris,5.0,13.5,11.5,7.45,16.76,14.81,0.691,0.699,0.689,over,over,over,0,79.89,0.160,Low,High,High
6713,Devin Vassell,Andrew Nembhard,Isaiah Jackson,17.5,17.5,7.5,14.77,20.73,10.10,0.661,0.686,0.681,under,over,over,0,66.77,0.134,High,High,Med
2194,Keyonte George,Ausar Thompson,Duncan Robinson,20.5,11.5,10.5,23.03,13.77,12.84,0.651,0.639,0.649,over,over,over,0,45.88,0.092,High,High,High


In [18]:
# df = playerScoring('Trey Murphy III', s26, current_date, teamStarPlayer, projectedStartingFive)
# playerContext('Trey Murphy III', s26, current_date, projectedStartingFive, mainStartingFive, teamStarPlayer)